# UWB Noise Discovery Lab

Read DS-TWR sessions *before* jumping to conclusions: drift, bursts, heavy tails, correlated noise,
dropouts. Plots are hints, not verdicts.

**Usage:** set `SESSION` below (a folder in `data/sessions/`, an `.ndjson` file, or `None` for the newest
non-trivial session) and run all. Works without `EXPECTED_DISTANCE_M`: noise is then measured around the
session mean, and bias only appears once a true distance is known.

Raw data is never modified: `raw_m` is the measurement, `corrected_m` is derived by the host.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---- configuration ---------------------------------------------------------------------------------
SESSION: str | None = None  # e.g. "261031-1956-experiment", or "data/fixtures/sample_session.ndjson"
EXPECTED_DISTANCE_M: float | None = None  # true distance; falls back to the session manifest if present
SIGNAL = "raw_m"  # which column to analyse: "raw_m" or "corrected_m"
ROLLING_WINDOW = 20  # samples
OUTLIER_Z = 4.0  # robust z-score (MAD) above which a sample is flagged
MIN_SAMPLES = 30  # below this, statistics are printed with a warning
# ----------------------------------------------------------------------------------------------------

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CLAUDE.md").exists())
SESSIONS_DIR = ROOT / "data" / "sessions"


def resolve_session(spec: str | None) -> Path:
    if spec is None:
        candidates = [
            d / "telemetry.ndjson"
            for d in sorted(SESSIONS_DIR.glob("*-experiment*"))
            if (d / "telemetry.ndjson").is_file()
        ]
        candidates = [c for c in candidates if c.stat().st_size > 0]
        if not candidates:
            return ROOT / "data" / "fixtures" / "sample_session.ndjson"
        return max(candidates, key=lambda c: (sum(1 for _ in c.open("rb")), c.stat().st_mtime))
    p = Path(spec)
    p = p if p.is_absolute() else (ROOT / p if (ROOT / p).exists() else SESSIONS_DIR / p)
    return p / "telemetry.ndjson" if p.is_dir() else p


SESSION_PATH = resolve_session(SESSION)
manifest_path = SESSION_PATH.parent / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
print(f"Session : {SESSION_PATH.relative_to(ROOT)}")
print(f"Manifest: { {k: manifest.get(k) for k in ('name', 'source', 'started_at', 'completed')} }")

Session : data/sessions/20261028-155158-experiment-88f991c5/telemetry.ndjson
Manifest: {'name': 'experiment', 'source': 'dual-serial', 'started_at': '2026-10-28T15:51:58.875756+00:00', 'completed': True}


In [2]:
REQUIRED = {"type", "v", "board", "peer", "seq", "time_ms", "raw_m", "corrected_m", "valid"}
OPTIONAL = [
    "host_time", "radio_revision", "ipatov_power", "ipatov_f1", "ipatov_f2", "ipatov_f3",
    "ipatov_fp_index", "ipatov_accum_count", "cia_diag1",
]  # fmt: skip


def load_range_ndjson(path: Path) -> tuple[pd.DataFrame, int, int]:
    """Parse range records; any other line (logs, blanks, bad JSON, wrong schema) counts as ignored."""
    records: list[dict] = []
    ignored = total = 0
    with path.open(encoding="utf-8") as f:
        for line in f:
            total += 1
            try:
                msg = json.loads(line)
                if msg.get("type") != "range" or not REQUIRED <= msg.keys():
                    raise ValueError
                rec = {
                    "v": int(msg["v"]), "board": int(msg["board"]), "peer": int(msg["peer"]),
                    "seq": int(msg["seq"]), "time_ms": int(msg["time_ms"]),
                    "raw_m": float(msg["raw_m"]), "corrected_m": float(msg["corrected_m"]),
                    "valid": bool(msg["valid"]),
                }  # fmt: skip
            except (ValueError, TypeError, AttributeError):
                ignored += 1
                continue
            rec.update({k: msg[k] for k in OPTIONAL if k in msg})
            records.append(rec)
    if not records:
        raise ValueError(f"No valid range records found in {path}")
    df = pd.DataFrame(records).sort_values("seq", kind="stable").reset_index(drop=True)
    if "host_time" in df:
        df["host_time"] = pd.to_datetime(df["host_time"], errors="coerce", utc=True)
    return df, ignored, total


df, ignored_lines, total_lines = load_range_ndjson(SESSION_PATH)
n_dupes = int(df["seq"].duplicated().sum())
df = df.drop_duplicates("seq", keep="first").reset_index(drop=True)
df["time_s"] = (df["time_ms"] - df["time_ms"].iloc[0]) / 1000.0
df.head()

,v,board,peer,seq,time_ms,raw_m,corrected_m,valid,host_time,radio_revision,time_s
0,1,1,2,14109,2184075,0.8090,0.8090,True,2026-10-28 15:51:58.944057+00:00,9,0.0
1,1,1,2,14110,2184175,0.7812,0.7812,True,2026-10-28 15:51:59.112932+00:00,9,0.1
2,1,1,2,14111,2184275,0.7997,0.7997,True,2026-10-28 15:51:59.281473+00:00,9,0.2
3,1,1,2,14112,2184375,0.8108,0.8108,True,2026-10-28 15:51:59.450633+00:00,9,0.3
4,1,1,2,14113,2184475,0.7956,0.7956,True,2026-10-28 15:51:59.619751+00:00,9,0.4


## 1. Acquisition quality
If this section looks bad, every curve below deserves suspicion — acquisition, not modelling, is the
bottleneck.

In [3]:
seq_missing = int((df["seq"].diff().dropna().clip(lower=1) - 1).sum())
dt = df["time_ms"].diff().dropna()
n_backsteps = int((dt <= 0).sum())
dt_pos = dt[dt > 0]
period_ms = float(dt_pos.median()) if len(dt_pos) else float("nan")

quality = pd.Series(
    {
        "lines in file": total_lines,
        "range records": len(df),
        "ignored lines": ignored_lines,
        "duplicate seq": n_dupes,
        "missing seq (dropouts)": seq_missing,
        "time_ms back-steps": n_backsteps,
        "valid ratio": round(float(df["valid"].mean()), 3),
        "median period [ms]": round(period_ms, 1),
        "rate [Hz]": round(1000 / period_ms, 2) if period_ms == period_ms else np.nan,
        "period jitter p95 [ms]": round(float(dt_pos.quantile(0.95)), 1) if len(dt_pos) else np.nan,
    },
    name="value",
)
if len(df) < MIN_SAMPLES:
    print(f"WARNING: only {len(df)} samples (< {MIN_SAMPLES}); treat every statistic below as anecdotal.")
quality.to_frame()

,value
lines in file,134.0
range records,134.0
ignored lines,0.0
duplicate seq,0.0
missing seq (dropouts),0.0
time_ms back-steps,0.0
valid ratio,1.0
median period [ms],100.0
rate [Hz],10.0
period jitter p95 [ms],100.0


In [4]:
# Analyse only valid samples; invalid ones stay in `df` and are shown in the overview.
d = df[df["valid"]].copy().reset_index(drop=True)
expected = EXPECTED_DISTANCE_M if EXPECTED_DISTANCE_M is not None else manifest.get("expected_m")
if expected is None and "expected_m" in manifest.get("config", {}):
    expected = manifest["config"]["expected_m"]

x = d[SIGNAL].to_numpy()
reference = float(expected) if expected is not None else float(np.median(x))
d["residual_m"] = x - reference
d["rolling_mean_m"] = d["residual_m"].rolling(ROLLING_WINDOW, min_periods=3).mean()
d["rolling_std_m"] = d["residual_m"].rolling(ROLLING_WINDOW, min_periods=3).std(ddof=1)

med = np.median(d["residual_m"])
mad = np.median(np.abs(d["residual_m"] - med)) * 1.4826
d["robust_z"] = (d["residual_m"] - med) / mad if mad > 0 else 0.0
d["outlier"] = d["robust_z"].abs() > OUTLIER_Z

label = (
    f"expected distance {reference:.3f} m" if expected is not None
    else f"session median {reference:.3f} m (no expected distance: bias unknown)"
)  # fmt: skip
print(f"Residuals of {SIGNAL} vs {label}; {int(d['outlier'].sum())} outliers at |z|>{OUTLIER_Z}")

Residuals of raw_m vs session median 0.795 m (no expected distance: bias unknown); 0 outliers at |z|>4.0


## 2. Overview

In [5]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df["time_s"], y=df["raw_m"], mode="lines", name="raw_m"))
fig.add_trace(go.Scatter(x=df["time_s"], y=df["corrected_m"], mode="lines", name="corrected_m"))
bad = df[~df["valid"]]
if len(bad):
    fig.add_trace(go.Scatter(x=bad["time_s"], y=bad["raw_m"], mode="markers", name="invalid",
                             marker=dict(color="red", symbol="x")))  # fmt: skip
out = d[d["outlier"]]
if len(out):
    fig.add_trace(go.Scatter(x=(out["time_ms"] - df["time_ms"].iloc[0]) / 1000, y=out[SIGNAL],
                             mode="markers", name="outlier", marker=dict(color="orange", size=9)))  # fmt: skip
if expected is not None:
    fig.add_hline(y=float(expected), line_dash="dash", annotation_text="expected")
fig.update_layout(title="Range overview", xaxis_title="time [s]", yaxis_title="distance [m]",
                  template="plotly_white", hovermode="x unified")  # fmt: skip
fig.show()

## 3. Distribution and temporal structure
Left→right: shape (Gaussian? heavy tail?), rolling mean/σ (drift, bursts), and autocorrelation
(white noise vs slow wander). Dashed lines in the histogram are the Gaussian with the same mean/σ.

In [6]:
r = d["residual_m"].to_numpy()
t = (d["time_ms"] - df["time_ms"].iloc[0]).to_numpy() / 1000.0

fig = make_subplots(rows=1, cols=3, subplot_titles=("Residual histogram", "Rolling mean ± σ", "Autocorrelation"))
fig.add_trace(go.Histogram(x=r, nbinsx=max(10, min(60, len(r) // 5)), histnorm="probability density",
                           name="residual"), row=1, col=1)  # fmt: skip
if len(r) > 2 and r.std() > 0:
    g = np.linspace(r.min(), r.max(), 200)
    pdf = np.exp(-0.5 * ((g - r.mean()) / r.std(ddof=1)) ** 2) / (r.std(ddof=1) * np.sqrt(2 * np.pi))
    fig.add_trace(go.Scatter(x=g, y=pdf, mode="lines", line=dict(dash="dash"), name="Gaussian fit"), row=1, col=1)

fig.add_trace(go.Scatter(x=t, y=d["residual_m"], mode="lines", line=dict(width=1, color="lightgray"),
                         name="residual"), row=1, col=2)  # fmt: skip
fig.add_trace(go.Scatter(x=t, y=d["rolling_mean_m"], mode="lines", name="rolling mean"), row=1, col=2)
fig.add_trace(go.Scatter(x=t, y=d["rolling_mean_m"] + d["rolling_std_m"], mode="lines",
                         line=dict(dash="dot"), name="+1σ"), row=1, col=2)  # fmt: skip
fig.add_trace(go.Scatter(x=t, y=d["rolling_mean_m"] - d["rolling_std_m"], mode="lines",
                         line=dict(dash="dot"), name="-1σ"), row=1, col=2)  # fmt: skip


def autocorr(v: np.ndarray, max_lag: int) -> np.ndarray:
    v = v - v.mean()
    denom = float(np.dot(v, v))
    if denom == 0:
        return np.zeros(max_lag + 1)
    return np.array([np.dot(v[: len(v) - k], v[k:]) / denom for k in range(max_lag + 1)])


max_lag = min(50, len(r) // 2)
acf = autocorr(r, max_lag) if max_lag >= 1 else np.array([1.0])
fig.add_trace(go.Bar(x=list(range(len(acf))), y=acf, name="ACF"), row=1, col=3)
if len(r) > 2:
    band = 1.96 / np.sqrt(len(r))  # 95% band for white noise
    fig.add_hrect(y0=-band, y1=band, fillcolor="gray", opacity=0.2, line_width=0, row=1, col=3)
fig.update_xaxes(title_text="residual [m]", row=1, col=1)
fig.update_xaxes(title_text="time [s]", row=1, col=2)
fig.update_xaxes(title_text="lag [samples]", row=1, col=3)
fig.update_layout(template="plotly_white", height=420, showlegend=False)
fig.show()

## 4. Allan deviation
Averaging N samples shrinks white noise like 1/√N. Where the curve flattens or rises, slow drift
dominates and more averaging stops helping — that horizon is a key modelling number.

In [7]:
def allan_deviation(v: np.ndarray, taus: list[int]) -> list[float]:
    out = []
    for m in taus:
        n = len(v) // m
        if n < 3:
            out.append(float("nan"))
            continue
        means = v[: n * m].reshape(n, m).mean(axis=1)
        out.append(float(np.sqrt(0.5 * np.mean(np.diff(means) ** 2))))
    return out


if len(r) >= 8:
    taus = sorted({int(round(m)) for m in np.geomspace(1, max(1, len(r) // 3), 15)})
    adev = allan_deviation(r, taus)
    tau_s = [m * (period_ms / 1000 if period_ms == period_ms else 1.0) for m in taus]
    ref = [adev[0] / np.sqrt(m) for m in taus]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=tau_s, y=adev, mode="lines+markers", name="Allan deviation"))
    fig.add_trace(go.Scatter(x=tau_s, y=ref, mode="lines", line=dict(dash="dash"), name="white-noise 1/√N"))
    fig.update_xaxes(type="log", title_text="averaging time τ [s]")
    fig.update_yaxes(type="log", title_text="σ [m]")
    fig.update_layout(template="plotly_white", title="Allan deviation of residual")
    fig.show()
else:
    print("Too few valid samples for an Allan deviation.")

## 5. Radio diagnostics (v2 telemetry)
Only shown when the session carries diagnostic fields. Residual-vs-diagnostic scatters are the first
place to look for LOS/NLOS signatures.

In [8]:
diag_cols = [c for c in OPTIONAL[2:] if c in d.columns and pd.api.types.is_numeric_dtype(d[c])]
if not diag_cols:
    print("This session has only core range fields (no diagnostics columns).")
else:
    fig = make_subplots(rows=len(diag_cols), cols=2, column_widths=[0.65, 0.35],
                        subplot_titles=[s for c in diag_cols for s in (c, f"|residual| vs {c}")])  # fmt: skip
    for i, c in enumerate(diag_cols, start=1):
        fig.add_trace(go.Scatter(x=t, y=d[c], mode="lines", name=c), row=i, col=1)
        fig.add_trace(go.Scatter(x=d[c], y=d["residual_m"].abs(), mode="markers", opacity=0.6, name=c), row=i, col=2)
    fig.update_layout(template="plotly_white", height=260 * len(diag_cols), showlegend=False)
    fig.show()
    print(d[diag_cols + ["residual_m"]].corr()["residual_m"].drop("residual_m").rename("corr with residual"))

This session has only core range fields (no diagnostics columns).


## 6. Summary

In [9]:
def stats(v: np.ndarray, truth_known: bool) -> dict[str, float]:
    s = float(v.std(ddof=1)) if len(v) > 1 else float("nan")
    centred = v - v.mean()
    kurt = float(np.mean(centred**4) / np.var(v) ** 2 - 3) if len(v) > 3 and np.var(v) > 0 else float("nan")
    skew = float(np.mean(centred**3) / np.std(v) ** 3) if len(v) > 2 and np.std(v) > 0 else float("nan")
    return {
        "n": len(v),
        "bias [m]": float(v.mean()) if truth_known else float("nan"),
        "std [m]": s,
        "rmse [m]": float(np.sqrt(np.mean(v**2))) if truth_known else float("nan"),
        "MAD-σ [m]": float(np.median(np.abs(v - np.median(v))) * 1.4826),
        "p95 |err| [m]": float(np.percentile(np.abs(v if truth_known else centred), 95)),
        "skew": skew,
        "excess kurtosis": kurt,
        "outliers": int(d["outlier"].sum()),
        "lag-1 autocorr": float(acf[1]) if len(acf) > 1 else float("nan"),
    }


summary = pd.DataFrame({"raw_m": stats(d["raw_m"].to_numpy() - reference, expected is not None),
                        "corrected_m": stats(d["corrected_m"].to_numpy() - reference, expected is not None)})  # fmt: skip
summary.round(4)

,raw_m,corrected_m
n,134.0000,134.0000
bias [m],NaN,NaN
std [m],0.0122,0.0122
rmse [m],NaN,NaN
MAD-σ [m],0.0128,0.0128
p95 |err| [m],0.0249,0.0249
skew,0.0770,0.0770
excess kurtosis,0.0072,0.0072
outliers,0.0000,0.0000
lag-1 autocorr,-0.0492,-0.0492


**Reading guide**
- `excess kurtosis` >> 0 or many outliers: heavy tails / NLOS-like bursts — don't fit a plain Gaussian.
- `lag-1 autocorr` well above ~`2/√n`: noise is not white; consecutive samples aren't independent.
- Allan curve leaving the 1/√N line: drift; pair it with the rolling mean plot.
- `MAD-σ` ≪ `std`: a few outliers inflate the standard deviation.
- Bias and RMSE are `NaN` until `EXPECTED_DISTANCE_M` (or the manifest) provides ground truth.

**Next steps:** compare sessions by condition (distance, LOS/NLOS, orientation), add Welch /
Lomb-Scargle spectra consistent with the host analysis, and export reproducible subsets for modelling.